# Droplet rebound simulation (SLA experiments) - Run 2D

This notebook is used to run the actual simulations for the droplet rebound test case. The simulation will be restarted from a previously computed initial state, see `DropletReboundInit.ipynb`

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using MPI.Wrappers;
using NUnit.Framework;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

In [ ]:
BoSSSshell.WorkflowMgm.Init("DropletRebound", myBatch);

In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

## Setting up restart

Choose if this notebook should do a restart. 

In [ ]:
bool restartRun = false;

In [ ]:
//OpenOrCreateDatabase(@"\\hpccluster\hpccluster-scratch\smuda\DropletRebound");
//OpenOrCreateDatabase(@"\\dc3\userspace\smuda\hpccluster\DropletRebound");
//OpenOrCreateDatabase(@"\\dc1\scratch\Smuda\exchange_db");

In [ ]:
var sessions = databases.Pick(0).Sessions;
sessions

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions

In [ ]:
//BoSSSshell.WorkflowMgm.Sessions.Pick(0).Delete(true)
//sessions.Pick(2).Copy(databases.Pick(2))

In [ ]:
//var restartSession = sessions.Where(sess => sess.Name.Contains("dropVelocity100vH")).Single();
var restartSession = databases.Pick(0).Sessions.Pick(0);
restartSession

In [ ]:
// List<ISessionInfo> restartSessionList = new List<ISessionInfo>();
// restartSessionList.Add(restartSession);
// Guid restartID = restartSession.RestartedFrom;
// while(restartID != Guid.Empty) {
//     var rSess = sessions.Where(sess => sess.ID == restartID).Single();
//     restartSessionList.Add(rSess);
//     restartID = rSess.RestartedFrom;
// }
// restartSessionList

In [ ]:
//restartSession.GetSessionDirectory()
//restartSession.Copy(databases.Pick(1));

In [ ]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(2).Do()

automated naming of restart sessions

In [ ]:
Stack<ISessionInfo>  procSIs = new Stack<ISessionInfo>();
procSIs.Push(restartSession);
var currSI = restartSession;
var rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
while(!rSIs.IsNullOrEmpty()) {
    var rSI = rSIs.Single();
    procSIs.Push(rSI);
    currSI = rSI;
    rSIs = sessions.Where(sess => sess.ID.Equals(currSI.RestartedFrom));
}
int restartNum = procSIs.Count;

string orgName = restartSession.Name;
string restartName;
if (restartNum == 1) {
    //restartName = orgName.Substring(0, orgName.Length - 10); // remoinvg _InitState
//} else if (restartNum == 1){
    restartName = orgName.Substring(0, orgName.Length) + "_restart"; // + restartNum; 
} else {
    restartName = orgName.Substring(0, orgName.Length - 1) + restartNum; 
}

restartName

In [ ]:
//restartName = "DropletRebound_8x8x8AMR1_k3_dropVelocity100vH_NeKs_RI1_halfdt_c4t4_restart4_checkUpdateFix"

## Boundary Conditions

0: pressure outlet, 1: velocity inlet

In [ ]:
static int BC_tag2_top = 0;

## Grid creation

The used grid is a cartesain box around the droplet injection location, which is located at `radiusOP` (in $r$-direction) away from the center of the rotating disk.

In [ ]:
static public Grid2D RotatingDiskSlice_CartesianCutOut(double radiusOP, double l_upstream, double l_downstream, double h_axial, int res_azimuthal, int res_axial) {

    double[] yNodes = GenericBlas.Linspace(-l_upstream, l_downstream, res_azimuthal + 1);    // azimuthal direction
    // double[] yNodes = GenericBlas.Linspace(-l_azimuthal / 3.0, (2.0 * l_azimuthal) / 3.0, res_azimuthal + 1);    // azimuthal direction
    double[] zNodes = GenericBlas.Linspace(0.0, h_axial, res_axial + 1);    // axial direction
    
    var grd = Grid2D.Cartesian2DGrid(yNodes, zNodes, periodicY: false);
    grd.Name = $"RotatingDiskSlice2D_CartesianCutOut_{res_azimuthal}x{res_axial}";

    grd.EdgeTagNames.Add(1, "velocity_inlet_rotatingDisk");

    if (BC_tag2_top == 0)
        grd.EdgeTagNames.Add(2, "pressure_outlet_top");
    if (BC_tag2_top == 1)
        grd.EdgeTagNames.Add(2, "velocity_inlet_top");
   
    grd.EdgeTagNames.Add(3, "velocity_inlet_upstream");
    grd.EdgeTagNames.Add(4, "pressure_outlet_downstream");

    grd.DefineEdgeTags(delegate (Vector X) {
        byte et = 0;
        if (X.y.Abs() <= 1e-8)
            et = 1;
        if ((X.y - h_axial).Abs() <= 1e-8)
            et = 2;
        if ((X.x + (l_upstream)).Abs() <= 1e-8)
            et = 3;
        if ((X.x - (l_downstream)).Abs() <= 1e-8)
            et = 4;

        return et;
    });    

    return grd;
}

## Simulation setup

In [ ]:
double radiusOP = 78e-3; // operating point (droplet injection point) -> Re = radiusOp / Lstar
double density = 1.19;  
double kinViscosity = 1.52e-5; // kinematic viscosity
//double omega = 46.11514476; // rotation rate
double omega = 1820.5128205128206; //46.11514476; // rotation rate
double Lstar = Math.Sqrt(kinViscosity / omega);  // used for non-dimensionalization of the flow fields
Lstar

Reynolds number at the point of injection

In [ ]:
double reynolds = radiusOP / Lstar;
reynolds

Boundary layer (BL) thickness

In [ ]:
double zBL = 5.5;
double zBLstar = zBL * Lstar;
zBLstar

Height of the computational domain (roughly 3 times the BL thickness)

In [ ]:
double zTop = 15;
double zTopStar = zTop * Lstar;
zTopStar

## Prescribed boundary conditions for the flow field 

The B.C. are given by the Homotopy Analysis method (HAM), which give a linear combination in term of $\sum_{n,i} \textrm{e}^{-n \eta} \eta^{i}$, where $\eta$ describes the dimensionless height in axial-direction.

In [ ]:
using BoSSS.Application.XNSE_Solver.SpecificSolutions;

In [ ]:
MultidimensionalArray HAMcoeff_velV = IMatrixExtensions.LoadFromTextFile($"vonKarmanFlowSolution_HAMcoeffV.txt");

In [ ]:
var vonKarmanHAM_velX = new RotatingDiskBoundaryLayerConditionsHAM_VelocityX2D();
vonKarmanHAM_velX.SetData(HAMcoeff_velV, omega, kinViscosity, radiusOP);

### B.C. for the rotating disk

One may also use the B.C. for the analytic solution

In [ ]:
Formula RotatingDiskVelocityX = new Formula(
    "VelX",
    false,
    "double VelX(double[] X) { " + 
    "double omega = 1820.5128205128206; "  + 
    "double radiusOP = 78e-3;; " + 
    "double velX = radiusOP * omega;" +
    "return velX; } "
);

### B.C. for the top boundary 

Again, one may use the B.C. for the analytic solution

In [ ]:
// double velW_top = -0.88447342;
// double velWstar_top = velW_top * Math.Sqrt(kinViscosity * omega); 
// velWstar_top

In [ ]:
// Formula VelocityZ_top = new Formula(
//     "VelZ",
//     false,
//     "double VelZ(double[] X) { return -0.14713075072584397; } "
// );

## Initial conditions for the injected droplet 

If not restarted from precomputed initial state with impact velocity for the droplet.

In [ ]:
Formula PhiFunc = new Formula(
    "Phi",
    false,
    "double Phi(double[] X) { " +
    "double radiusOP = 78e-3;" +
    "double radiusDrop = 0.1872e-3;" +   // changed from 0.205e-3
    "double initHeight = 45e-5;" +  // changed from 75e-5
    "return Math.Sqrt(X[0].Pow2() + (X[1] - initHeight).Pow2()) - radiusDrop; } "
);

In [ ]:
Formula InitVelocity = new Formula(
    "VelY",
    false,
    "double VelY(double[] X) { return -1.12; } "
);

In [ ]:
Formula GravityValue = new Formula(
    "GravY",
    false,
    "double GravY(double[] X) { return -9.81; } "
);

## Control settings

In [ ]:
// set grid 
int res_global = 8;

double l_upstream = zTopStar * (2.0 / 2.0);
double l_downstream = zTopStar * (4.0 / 2.0);
double h_axial = 2.0 * zTopStar; 

int res_azimuthal = 3 * res_global;
int res_axial = 2 * res_global;

In [ ]:
double densityDrop = 998.37;
double sigma = 72.9e-3;

int res_global = 8;
int AMRlevel_disk = 1;
int AMRlevel_drop = 2;
int AMRlevel_dropBL = 0;
int maxAMRlevel = (new int[] {AMRlevel_disk, AMRlevel_drop, AMRlevel_dropBL}).Max();
double hmin = (zTopStar / res_global) / (maxAMRlevel + 1);

int k = 3;

double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(densityDrop, density, sigma, hmin, k+1);
dt_sigma

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();
Controls.Clear();

In [ ]:
var C = new XNSE_Control();


C.SetDGdegree(k);
C.FieldOptions.Add(VariableNames.GravityY, new FieldOpts() {
    SaveToDB = FieldOpts.SaveToDBOpt.TRUE
});


// physical parameters
C.PhysicalParameters.rho_A = densityDrop;
C.PhysicalParameters.mu_A = densityDrop * 0.000001028;

C.PhysicalParameters.rho_B = density;
C.PhysicalParameters.mu_B = density * kinViscosity;

C.PhysicalParameters.Sigma = sigma;

C.PhysicalParameters.IncludeConvection = true;

    
if (restartRun) {
    C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, 350);

} else {
    // set grid
    Grid2D grd = RotatingDiskSlice_CartesianCutOut(radiusOP, l_upstream, l_downstream, h_axial, res_azimuthal, res_axial);
    C.SetGrid(grd);

    // initial conditions
    C.AddInitialValue("VelocityX#B", vonKarmanHAM_velX);

    C.AddInitialValue("Phi", PhiFunc);
    C.AddInitialValue("VelocityY#A", InitVelocity);

}

C.AddInitialValue("GravityY#A", GravityValue);
C.AddInitialValue("GravityY#B", GravityValue);

// boundary conditions
C.AddBoundaryValue("velocity_inlet_rotatingDisk", "VelocityX#B", RotatingDiskVelocityX);

if (BC_tag2_top == 1) {
    C.AddBoundaryValue("velocity_inlet_top", "VelocityX#B", vonKarmanHAM_velX);
}

C.AddBoundaryValue("velocity_inlet_upstream", "VelocityX#B", vonKarmanHAM_velX);

// C.AddBoundaryValue("velocity_inlet_downstream", "VelocityX#B", vonKarman_velX);


C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

C.ReInitPeriod = 20;

// C.SkipSolveAndEvaluateResidual = true;

// C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
C.NonLinearSolver.Globalization = Newton.GlobalizationOption.LineSearch;
// C.NonLinearSolver.ConvergenceCriterion = 1e-9;
C.NonLinearSolver.MaxSolverIterations = 50;

//C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


// C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
// C.Timestepper_LevelSetHandling = LevelSetHandling.None;
// C.Option_LevelSetEvolution = LevelSetEvolution.None;

C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
C.TimeSteppingScheme = TimeSteppingScheme.ImplicitEuler;
C.dtFixed = 5e-7;
C.NoOfTimesteps = 1000;
    
{
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel_drop });
    C.activeAMRlevelIndicators.Add(new AMRLevelIndicatorLibrary.AMRonBoundary(new byte[] { 1 }) { maxRefinementLevel = AMRlevel_disk });
    C.AMR_startUpSweeps = maxAMRlevel;
}

// C.PostprocessingModules.Add(new EnergyLogging());
C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers,BoSSS.Foundation.ConstrainedDGprojection";
    
if (restartRun) 
    C.SessionName = restartName;
else {
    C.SessionName = $"DropletReboundSLA_2D_{res_azimuthal}x{res_axial}AMR{maxAMRlevel}_k{k}_fastMarching_ReInit{C.ReInitPeriod}";
}
    
    
Controls.Add(C);

## Launch job

In [ ]:
Controls.Select(C => C.SessionName)

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    //IDictionary<string, string> args = oneJob.EnvironmentVars;
    //args["BOSSS_ARG_7"] = numThreads.ToString();
    //args.Remove("OMP_NUM_THREADS");
    //oneJob.MySetCommandLineArguments(args.Values.ToArray());
    //oneJob.UseComputeNodesExclusive = true;
    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch); 
}

In [ ]:
wmg.Sessions

In [ ]:
//wmg.Sessions.Pick(1).Timesteps.Take(5)

In [ ]:
wmg.Sessions.Pick(0).Export().WithSupersampling(2).Do();